In [124]:
import pandas as pd
import json
import os
from tqdm import tqdm
tqdm.pandas()

task = 'openset'
final_df = []
for method in os.listdir(f"results/{task}"):
    result_file = f"results/{task}/{method}/results.csv"
    df =pd.read_csv(result_file)
    df = df[~df['args'].isna()]
    df['args'] = df['args'].progress_apply(json.loads)
    df['fold_idx'] = df['args'].apply(lambda x: int(x['fold_idx']))
    df['num_train_epochs'] = df['args'].apply(lambda x: x['num_train_epochs'])
    # df['num_pretrain_epochs'] = df['args'].apply(lambda x: x['num_pretrain_epochs'])
    df = df[(df['num_train_epochs'].apply(int)==50)]
    df = df.drop(['cluster_num_factor', 'args', 'seed', 'num_train_epochs'], axis=1)
    df = df.drop_duplicates(['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'fold_idx'])
    final_df.append(df)

final_df = pd.concat(final_df)
final_df = final_df.set_index(['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'fold_idx'])
df_melted = final_df.stack().reset_index()
df_melted = df_melted.rename(columns={0: "value", "level_5": "metric"})
df_melted = df_melted.sort_values(['dataset', 'method', 'fold_idx', 'labeled_ratio', 'known_cls_ratio', 'metric'])
df_melted

100%|██████████| 79/79 [00:00<00:00, 70440.05it/s]


,labeled_ratio,known_cls_ratio,dataset,method,fold_idx,metric,value
296,0.1,0.25,20NG,ADB,0,ACC,53.8000
297,0.1,0.25,20NG,ADB,0,F1,50.7301
298,0.1,0.25,20NG,ADB,0,K-F1,48.3731
299,0.1,0.25,20NG,ADB,0,N-F1,62.5150
236,1.0,0.25,20NG,ADB,0,ACC,64.6900
...,...,...,...,...,...,...,...
591,0.1,0.50,thucnews,clap,0,N-F1,0.0000
616,0.1,0.75,thucnews,clap,0,ACC,50.3200
617,0.1,0.75,thucnews,clap,0,F1,54.4698
618,0.1,0.75,thucnews,clap,0,K-F1,0.0000


In [125]:
df_pivot = df_melted.pivot(index=['dataset', 'method', 'fold_idx'], columns=['labeled_ratio', 'known_cls_ratio', 'metric'], values='value')
df_pivot.to_excel(f'results/{task}_pivot.xlsx')
df_pivot

labeled_ratio                      0.1                                   \
known_cls_ratio                   0.25                                    
metric                             ACC         F1       K-F1       N-F1   
dataset       method  fold_idx                                            
20NG          ADB     0         53.800  50.730100  48.373100  62.515000   
                      3         51.450  56.462700  56.991000  53.821000   
              DOC     0          0.785   0.379602   0.280936   0.872931   
              DeepUnk 0          0.807   0.459434   0.374406   0.884569   
              DyEn    0          0.991   0.984290   0.982358   0.993952   
...                                ...        ...        ...        ...   
stackoverflow ab      0         73.700  58.100000  53.100000  83.200000   
                      3         80.000  65.400000  61.000000  87.000000   
              clap    0         31.110  47.271500   0.000000   0.000000   
thucnews      ab      0         32.700  26.700000  22.600000  43.100000   
              clap    0         22.940  32.559800   0.000000   0.000000   

labeled_ratio                     1.0                               0.1  \
known_cls_ratio                  0.25                              0.50   
metric                            ACC       F1     K-F1     N-F1    ACC   
dataset       method  fold_idx                                            
20NG          ADB     0         64.69  67.9614  67.5778  69.8795    NaN   
                      3           NaN      NaN      NaN      NaN    NaN   
              DOC     0           NaN      NaN      NaN      NaN    NaN   
              DeepUnk 0           NaN      NaN      NaN      NaN    NaN   
              DyEn    0           NaN      NaN      NaN      NaN    NaN   
...                               ...      ...      ...      ...    ...   
stackoverflow ab      0           NaN      NaN      NaN      NaN  66.90   
                      3           NaN      NaN      NaN      NaN    NaN   
              clap    0           NaN      NaN      NaN      NaN  43.97   
thucnews      ab      0           NaN      NaN      NaN      NaN  34.20   
              clap    0           NaN      NaN      NaN      NaN  35.19   

labeled_ratio                                                              \
known_cls_ratio                                       0.75                  
metric                               F1  K-F1  N-F1    ACC       F1  K-F1   
dataset       method  fold_idx                                              
20NG          ADB     0             NaN   NaN   NaN    NaN      NaN   NaN   
                      3             NaN   NaN   NaN    NaN      NaN   NaN   
              DOC     0             NaN   NaN   NaN    NaN      NaN   NaN   
              DeepUnk 0             NaN   NaN   NaN    NaN      NaN   NaN   
              DyEn    0             NaN   NaN   NaN    NaN      NaN   NaN   
...                                 ...   ...   ...    ...      ...   ...   
stackoverflow ab      0         63.4000  62.5  72.1  61.60  65.2000  66.0   
                      3             NaN   NaN   NaN    NaN      NaN   NaN   
              clap    0         59.0736   0.0   0.0  80.77  84.1890   0.0   
thucnews      ab      0         28.3000  26.2  43.5  27.10  24.8000  24.3   
              clap    0         43.5344   0.0   0.0  50.32  54.4698   0.0   

labeled_ratio                         1.0                
known_cls_ratio                      0.50                
metric                          N-F1  ACC  F1 K-F1 N-F1  
dataset       method  fold_idx                           
20NG          ADB     0          NaN  NaN NaN  NaN  NaN  
                      3          NaN  NaN NaN  NaN  NaN  
              DOC     0          NaN  NaN NaN  NaN  NaN  
              DeepUnk 0          NaN  NaN NaN  NaN  NaN  
              DyEn    0          NaN  NaN NaN  NaN  NaN  
...                              ...  ...  ..  ...  ...  
stackoverflo

In [126]:
# ====== 2) （可选）若有计划 fold 列表，显示“完成数/计划数” ======
# 例如计划 folds 为 [0,1,2,3,4]（共5个）
EXPECTED_FOLDS = None  # e.g., [0,1,2,3,4]
if EXPECTED_FOLDS is not None:
    plan_n = len(EXPECTED_FOLDS)
    # 统计每格完成的 fold 个数
    fold_done_cnt = fold_coverage.copy()
    fold_done_cnt["plan"] = plan_n
    fold_done_cnt["text"] = fold_done_cnt["folds_done"].astype(int).astype(str) + "/" + str(plan_n)
    pivot_folds_text = fold_done_cnt.pivot(
        index=["dataset", "method"],
        columns=["labeled_ratio", "known_cls_ratio"],
        values="text"
    ).sort_index()

# ====== 3) 每格显示“已完成的 fold 列表”，便于直观看缺哪几个 ======
# 先做一个按格聚合出 fold 列表的表
fold_list = (cells.groupby(["dataset","method","labeled_ratio","known_cls_ratio"])["fold_idx"]
             .apply(lambda s: sorted(pd.unique(s)))
             .reset_index(name="folds_list"))
fold_list["folds_list_str"] = fold_list["folds_list"].apply(lambda x: "[" + ", ".join(map(str, x)) + "]")

pivot_folds_list = fold_list.pivot(
    index=["dataset", "method"],
    columns=["labeled_ratio", "known_cls_ratio"],
    values="folds_list_str"
).sort_index()

pivot_folds_list.to_excel(f'results/{task}_progress.xlsx')
pivot_folds_list

labeled_ratio                  0.1                                       0.5  \
known_cls_ratio               0.25          0.50          0.75          0.25   
dataset  method                                                                
20NG     ALUP         [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         DPN          [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         DeepAligned  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         GeoID        [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         LOOP         [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
...                            ...           ...           ...           ...   
thucnews ALUP         [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         DPN          [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         PLM_GCD      [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         SDC                   NaN           [3]           [3]           [3]   
         TAN                   [0]           [0]           [0]           NaN   

labeled_ratio                                              1.0                \
known_cls_ratio               0.50          0.75          0.25          0.50   
dataset  method                                                                
20NG     ALUP         [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         DPN          [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         DeepAligned  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         GeoID        [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         LOOP         [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
...                            ...           ...           ...           ...   
thucnews ALUP         [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         DPN          [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         PLM_GCD      [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]  [0, 1, 2, 3]   
         SDC                   [3]           [3]           [3]           [3]   
         TAN                   NaN           NaN           NaN           NaN   

labeled_ratio                       
known_cls_ratio               0.75  
dataset  method                     
20NG     ALUP         [0, 1, 2, 3]  
         DPN          [0, 1, 2, 3]  
         DeepAligned  [0, 1, 2, 3]  
         GeoID        [0, 1, 2, 3]  
         LOOP         [0, 1, 2, 3]  
...                            ...  
thucnews ALUP         [0, 1, 2, 3]  
         DPN          [0, 1, 2, 3]  
         PLM_GCD      [0, 1, 2, 3]  
         SDC                   [3]  
         TAN                   NaN  

[83 rows x 9 columns]

In [127]:
all_num = df_melted['dataset'].nunique() * 3 * 3 * df_melted['method'].nunique() * 5
cur_num = len(df_melted) / df_melted['metric'].nunique()
cur_num / all_num

0.039141414141414144

In [128]:
all_num

3960

In [129]:
cur_num

155.0